In [6]:
import pandas as pd
import numpy as np

# LOADED THE MESSY DATA
df = pd.read_csv('marketing_campaign_data_messy.csv')

print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 2020 rows, 12 columns


In [7]:
# STEP 1: HEADER CLEANING

print(df.columns.tolist())
df.columns = df.columns.str.strip().str.lower().str.replace(' ', ' ')

print("----- FIX APPLIED -----")
print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
----- FIX APPLIED -----
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


In [8]:
# STEP 2: TYPE CONVERSION & CURRENCY CLEANING

dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask, ['campaign_id', 'spend']].head(3))

df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'])

print("----- FIX APPLIED -----")
print(df.loc[dirty_spend_mask, ['campaign_id', 'spend']].head(3)) # to compare

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
----- FIX APPLIED -----
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


In [9]:
# STEP 3: CATEGORICAL TYPOS (FUZZY LOGIC)

print(df['channel'].unique())

cleanup_map = {
    'Facebok': 'Facebook',
    'Insta_gram': 'Instagram',
    'Gogle': 'Google Ads',
    'Tik_Tok': 'TikTok',
    'E-mail': 'Email',
    'N/A': np.nan # Handling the ghost value here too

}

df['channel'] = df['channel'].replace(cleanup_map)

print("----- FIX APPLIED -----")
print(df['channel'].unique())

<StringArray>
[    'TikTok',   'Facebook',      'Email',  'Instagram', 'Google Ads',
     'E-mail',          nan,      'Gogle',    'Tik_Tok',    'Facebok',
 'Insta_gram']
Length: 11, dtype: str
----- FIX APPLIED -----
<StringArray>
['TikTok', 'Facebook', 'Email', 'Instagram', 'Google Ads', nan]
Length: 6, dtype: str


In [12]:
# STEP 4: HANDLING MIXED BOOLEANS

print(df['active'].unique())

bool_map = {
    'Yes': True,
    'Y': True,
    1: True,
    'No': False,
    '0': False,
    0: False,

}
# map the right values to what we want in the final version
df['active'] = df['active'].map(bool_map).fillna(False).astype(bool)

print('----- FIX APPLIED -----')
print(df['active'].unique())

<StringArray>
['Y', '0', 'No', 'True', 'Yes', '1', 'False']
Length: 7, dtype: str
----- FIX APPLIED -----
[ True False]


In [15]:
# STEP 5: DATE PARSING

print(df['start_date'].dtype)

df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')

print('----- FIX APPLIED -----')
print(df['start_date'].dtype)


datetime64[us]
----- FIX APPLIED -----
datetime64[us]


/var/folders/qt/fxj532751xl3b3kfhhp8c44r0000gn/T/ipykernel_1792/1311759951.py:6: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')


In [21]:
# STEP 6: LOGICAL INTEGRITY (CLICKS vs IMPRESSIONS)

df = df.loc[:, ~df.columns.duplicated()] #located the columns that are duplicated and removes it

impossible_mask = df['clicks'] > df['impressions']
print(df.loc[impossible_mask, ['campaign_id', 'impressions', 'clicks']].head(3))

Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


In [ ]:
# STEP 7: LOGICAL INTEGRITY (TIME TRAVEL)

# I wanna make sure we have kind of chronological order
# between start and end date and no instances
# where the end date is actually in the past compared to the start date

time_travel_mask = df['end_date'] < df['start_date']
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

df.loc[time_travel_mask, 'end_date'] = df.loc[time_travel_mask, 'start_date'] + pd.Timedelta(days=30)

print('----- FIX APPLIED -----')
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

# Basically fix the right time


Empty DataFrame
Columns: [campaign_id, start_date, end_date]
Index: []
----- FIX APPLIED -----
Empty DataFrame
Columns: [campaign_id, start_date, end_date]
Index: []


In [34]:
# STEP 8: HANDLING OUTLIERS (WINSORIZING)

Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

IQR = Q3 - Q1
upper_limit = Q3 + (3 + IQR)

outlier_mask = df['spend'] > upper_limit
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))

# Basically, calculating the interquartile range which is kind of like an idea here
# I wanna see what is the usual range for the value spend.
# So what is considered like a normal variation for this value

# saying get the Q3, and add 3 times the usual range

print('----- FIX APPLIED -----')
df.loc[outlier_mask, 'spend'] = upper_limit
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))


   campaign_id    spend
12   CMP-00013  5292.16
22   CMP-00023  4726.22
57   CMP-00058  5719.84
----- FIX APPLIED -----
   campaign_id      spend
12   CMP-00013  4629.7675
22   CMP-00023  4629.7675
57   CMP-00058  4629.7675


In [36]:
# STEP 9: STRING PARSING (FEATURE EXTRACTION)

print(df['campaign_name'].head(3))

df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')

print('----- FIX APPLIED -----')

print(df[['campaign_name', 'season']].head(3))

0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: str
----- FIX APPLIED -----
         campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter


In [37]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,season
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Summer
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA,Launch
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,False,TI,Summer
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO,Summer
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN,Launch
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA,Winter
